# Proyecto: Sistema de Monitoreo de Calidad del Aire - SIATA
## Análisis Predictivo y Prescriptivo del Material Particulado PM2.5 en el Valle de Aburrá

### 1. Situación Problema

**Contexto:** La Secretaría de Salud Municipal de Medellín requiere un sistema de apoyo a la decisión para identificar zonas críticas de contaminación por material particulado PM2.5 y emitir recomendaciones preventivas a la población vulnerable (niños, adultos mayores y personas con enfermedades respiratorias).

**Necesidad:** Consultar información actualizada de calidad del aire de las estaciones de monitoreo del SIATA (Sistema de Alerta Temprana de Medellín y el Valle de Aburrá) para:
- Identificar estaciones con niveles de riesgo
- Visualizar geográficamente la distribución del contaminante
- Generar alertas tempranas y recomendaciones prescriptivas
- Apoyar la toma de decisiones en salud pública

**Fuente de datos:** SIATA - Red de monitoreo de calidad del aire del Valle de Aburrá

**API:** https://siata.gov.co/EntregaData1/Datos_SIATA_Aire_AQ_pm25_Last.json


## 2. Marco Teórico: ¿Qué es el PM2.5 y el ICA?

**Material Particulado PM2.5:** Partículas suspendidas en el aire con diámetro menor a 2.5 micrómetros. Provienen de:
- Combustión de vehículos
- Industrias
- Quemas agrícolas
- Construcción

**Índice de Calidad del Aire (ICA):** Escala estandarizada que resume el nivel de contaminación:

| Rango ICA | Calificación | Color | Recomendación |
|-----------|--------------|-------|---------------|
| 0-50 | Buena | Verde | Actividades normales |
| 51-100 | Aceptable | Amarillo | Personas sensibles reducir actividad prolongada |
| 101-150 | Moderada | Naranja | Evitar ejercicio intenso al aire libre |
| 151-200 | Mala | Rojo | Usar mascarilla, limitar exposición |
| 201-300 | Muy Mala | Morado | Permanecer en interiores |
| 301+ | Peligrosa | Café | Activar alerta sanitaria |

SIATA monitorea 43 puntos de monitoreo entre automáticos y manuales en el Valle de Aburrá [[22]].


## 3. Metodología

1. **Consumo de API:** Conexión a SIATA y extracción de datos en tiempo real
2. **Limpieza y transformación:** Procesamiento de JSON, manejo de valores nulos
3. **Cálculo del ICA:** Conversión de µg/m³ a índice de calidad del aire
4. **Análisis exploratorio:** Estadísticas descriptivas, identificación de patrones
5. **Visualización geográfica:** Mapeo de estaciones con folium
6. **Modelo prescriptivo:** Generación de recomendaciones automáticas
7. **Aplicación interactiva:** Filtros por zona, consulta de estaciones
8. **Landing page:** Presentación de la solución y utilidad


In [ ]:
# Instalación de librerías
!pip install requests pandas matplotlib seaborn folium ipywidgets --quiet

In [ ]:
# Importación de librerías
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 4. Consumo de la API SIATA


In [ ]:
# Configuración de la API
SIATA_API_URL = "https://siata.gov.co/EntregaData1/Datos_SIATA_Aire_AQ_pm25_Last.json"

# Función para consumir la API
def consumir_api_siata(url=SIATA_API_URL):
    """
    Consume la API de SIATA y retorna los datos en formato JSON
    
    Parámetros:
    url (str): URL de la API
    
    Retorna:
    dict: Datos JSON de la API
    """
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()
        print(f"✓ API consumida exitosamente")
        print(f"  - Código de estado: {response.status_code}")
        print(f"  - Fecha de consulta: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        return data
    except requests.exceptions.RequestException as e:
        print(f"✗ Error al consumir la API: {e}")
        return None

# Consumir la API
datos_siata = consumir_api_siata()

## 5. Exploración de la Estructura JSON


In [ ]:
# Inspeccionar la estructura de datos
if datos_siata:
    print("=" * 70)
    print("EXPLORACIÓN DE LA ESTRUCTURA JSON")
    print("=" * 70)
    print(f"\nTipo de dato: {type(datos_siata)}")
    print(f"Claves principales: {datos_siata.keys() if isinstance(datos_siata, dict) else 'Es una lista'}")
    
    if 'measurements' in datos_siata:
        print(f"\nNúmero de estaciones: {len(datos_siata['measurements'])}")
        print(f"\nEstructura de una medición:")
        print(json.dumps(datos_siata['measurements'][0], indent=2, ensure_ascii=False))

## 6. Transformación a DataFrame


In [ ]:
# Función para transformar JSON a DataFrame
def transformar_a_dataframe(data):
    """
    Transforma los datos JSON de SIATA a un DataFrame de pandas
    
    Parámetros:
    data (dict): Datos JSON de la API
    
    Retorna:
    pd.DataFrame: DataFrame con los datos procesados
    """
    if not data or 'measurements' not in data:
        print("No hay datos para transformar")
        return pd.DataFrame()
    
    # Extraer mediciones
    mediciones = data['measurements']
    
    # Crear lista de diccionarios aplanados
    datos_procesados = []
    
    for med in mediciones:
        registro = {
            'estacion': med.get('location', 'N/A'),
            'ciudad': med.get('city', 'N/A'),
            'pais': med.get('country', 'N/A'),
            'latitud': med.get('coordinates', {}).get('latitude', None),
            'longitud': med.get('coordinates', {}).get('longitude', None),
            'pm25': med.get('value', None),
            'unidad': med.get('unit', 'µg/m³'),
            'parametro': med.get('parameter', 'pm25'),
            'fecha_utc': med.get('date', {}).get('utc', None),
            'fecha_local': med.get('date', {}).get('local', None),
            'periodo_promedio_horas': med.get('averagingPeriod', {}).get('value', 1),
            'fuente': med.get('sourceName', 'SIATA'),
            'tipo_fuente': med.get('sourceType', 'government')
        }
        datos_procesados.append(registro)
    
    # Crear DataFrame
    df = pd.DataFrame(datos_procesados)
    
    # Convertir fechas
    df['fecha_utc'] = pd.to_datetime(df['fecha_utc'], errors='coerce')
    df['fecha_local'] = pd.to_datetime(df['fecha_local'], errors='coerce')
    
    return df

# Transformar datos
df_siata = transformar_a_dataframe(datos_siata)

# Mostrar primeras filas
if not df_siata.empty:
    print(f"\n✓ DataFrame creado: {len(df_siata)} estaciones")
    display(df_siata.head())

## 7. Exploración y Limpieza de Datos


In [ ]:
# Análisis exploratorio de datos
if not df_siata.empty:
    print("=" * 70)
    print("ANÁLISIS EXPLORATORIO DE DATOS")
    print("=" * 70)
    
    print(f"\n1. DIMENSIÓN DEL DATASET")
    print(f"   Filas: {df_siata.shape[0]}")
    print(f"   Columnas: {df_siata.shape[1]}")
    
    print(f"\n2. TIPOS DE DATOS")
    print(df_siata.dtypes)
    
    print(f"\n3. VALORES NULOS")
    nulos = df_siata.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "   No hay valores nulos")
    
    print(f"\n4. ESTADÍSTICAS DESCRIPTIVAS - PM2.5")
    # Filtrar valores válidos (excluir -9999 que son datos inválidos)
    pm25_validos = df_siata[(df_siata['pm25'] != -9999) & (df_siata['pm25'].notna())]['pm25']
    print(pm25_validos.describe())
    
    print(f"\n5. RANGO DE PM2.5")
    print(f"   Mínimo: {pm25_validos.min():.2f} µg/m³")
    print(f"   Máximo: {pm25_validos.max():.2f} µg/m³")
    print(f"   Media: {pm25_validos.mean():.2f} µg/m³")
    print(f"   Mediana: {pm25_validos.median():.2f} µg/m³")
    
    print(f"\n6. ESTACIONES CON DATOS INVÁLIDOS (-9999)")
    invalidas = df_siata[df_siata['pm25'] == -9999]['estacion'].tolist()
    print(f"   Cantidad: {len(invalidas)}")
    if invalidas:
        print(f"   Estaciones: {', '.join(invalidas[:5])}{'...' if len(invalidas) > 5 else ''}")

## 8. Cálculo del Índice de Calidad del Aire (ICA)


In [ ]:
# Función para calcular ICA basado en PM2.5
def calcular_ica_pm25(valor_pm25):
    """
    Calcula el Índice de Calidad del Aire (ICA) basado en la concentración de PM2.5
    Según estándares EPA y SIATA
    
    Parámetros:
    valor_pm25 (float): Concentración de PM2.5 en µg/m³
    
    Retorna:
    dict: Diccionario con ICA, categoría, color y recomendación
    """
    if pd.isna(valor_pm25) or valor_pm25 == -9999:
        return {
            'ica': None,
            'categoria': 'Sin dato',
            'color': 'gray',
            'recomendacion': 'No hay datos disponibles'
        }
    
    # Tabla de conversión PM2.5 a ICA (basado en EPA/SIATA)
    if valor_pm25 <= 12:
        ica = (50 / 12) * valor_pm25
        categoria = 'Buena'
        color = 'green'
        recomendacion = 'Calidad del aire satisfactoria. Actividades normales.'
    elif valor_pm25 <= 35.4:
        ica = 50 + ((100 - 50) / (35.4 - 12)) * (valor_pm25 - 12)
        categoria = 'Aceptable'
        color = 'yellow'
        recomendacion = 'Personas sensibles reducir actividad prolongada al aire libre.'
    elif valor_pm25 <= 55.4:
        ica = 100 + ((150 - 100) / (55.4 - 35.4)) * (valor_pm25 - 35.4)
        categoria = 'Moderada'
        color = 'orange'
        recomendacion = 'Evitar ejercicio intenso al aire libre.'
    elif valor_pm25 <= 150.4:
        ica = 150 + ((200 - 150) / (150.4 - 55.4)) * (valor_pm25 - 55.4)
        categoria = 'Mala'
        color = 'red'
        recomendacion = 'Usar mascarilla y limitar exposición al aire libre.'
    elif valor_pm25 <= 250.4:
        ica = 200 + ((300 - 200) / (250.4 - 150.4)) * (valor_pm25 - 150.4)
        categoria = 'Muy Mala'
        color = 'purple'
        recomendacion = 'Permanecer en interiores. Activar alerta.'
    else:
        ica = 300 + ((500 - 300) / (500.4 - 250.4)) * (valor_pm25 - 250.4)
        categoria = 'Peligrosa'
        color = 'maroon'
        recomendacion = 'Emergencia sanitaria. Evitar cualquier exposición.'
    
    return {
        'ica': round(ica, 1),
        'categoria': categoria,
        'color': color,
        'recomendacion': recomendacion
    }

# Aplicar cálculo de ICA al DataFrame
if not df_siata.empty:
    # Calcular ICA para cada estación
    resultados_ica = df_siata['pm25'].apply(calcular_ica_pm25)
    
    # Agregar columnas al DataFrame
    df_siata['ica'] = resultados_ica.apply(lambda x: x['ica'])
    df_siata['categoria'] = resultados_ica.apply(lambda x: x['categoria'])
    df_siata['color'] = resultados_ica.apply(lambda x: x['color'])
    df_siata['recomendacion'] = resultados_ica.apply(lambda x: x['recomendacion'])
    
    print("✓ ICA calculado para todas las estaciones")
    print(f"\nDistribución por categoría:")
    print(df_siata['categoria'].value_counts())

## 9. Visualización de Datos


In [ ]:
# Visualización 1: Distribución de PM2.5
if not df_siata.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Filtrar datos válidos
    df_validos = df_siata[(df_siata['pm25'] != -9999) & (df_siata['pm25'].notna())]
    
    # Gráfico 1: Histograma de PM2.5
    axes[0, 0].hist(df_validos['pm25'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 0].axvline(df_validos['pm25'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_validos["pm25"].mean():.2f}')
    axes[0, 0].axvline(df_validos['pm25'].median(), color='green', linestyle='--', linewidth=2, label=f'Mediana: {df_validos["pm25"].median():.2f}')
    axes[0, 0].set_xlabel('PM2.5 (µg/m³)')
    axes[0, 0].set_ylabel('Frecuencia')
    axes[0, 0].set_title('Distribución de Concentraciones PM2.5')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Gráfico 2: Distribución por categoría ICA
    categorias = df_siata['categoria'].value_counts()
    colores_categorias = {'Buena': 'green', 'Aceptable': 'yellow', 'Moderada': 'orange', 
                          'Mala': 'red', 'Muy Mala': 'purple', 'Peligrosa': 'maroon', 'Sin dato': 'gray'}
    colores_barras = [colores_categorias.get(cat, 'gray') for cat in categorias.index]
    
    axes[0, 1].bar(categorias.index, categorias.values, color=colores_barras, edgecolor='black', alpha=0.7)
    axes[0, 1].set_xlabel('Categoría ICA')
    axes[0, 1].set_ylabel('Número de Estaciones')
    axes[0, 1].set_title('Distribución por Categoría de Calidad del Aire')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Gráfico 3: Top 10 estaciones con mayor PM2.5
    top_estaciones = df_validos.nlargest(10, 'pm25')[['estacion', 'pm25']]
    ejes_y = range(len(top_estaciones))
    axes[1, 0].barh(ejes_y, top_estaciones['pm25'], color='coral', edgecolor='black', alpha=0.7)
    axes[1, 0].set_yticks(ejes_y)
    axes[1, 0].set_yticklabels([est[:30] + '...' if len(est) > 30 else est for est in top_estaciones['estacion']])
    axes[1, 0].set_xlabel('PM2.5 (µg/m³)')
    axes[1, 0].set_title('Top 10 Estaciones con Mayor Contaminación')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Gráfico 4: Boxplot de PM2.5
    axes[1, 1].boxplot(df_validos['pm25'], vert=True, patch_artist=True, labels=['PM2.5'])
    axes[1, 1].set_ylabel('PM2.5 (µg/m³)')
    axes[1, 1].set_title('Distribución Estadística de PM2.5')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Estadísticas resumen
    print("=" * 70)
    print("RESUMEN DE VISUALIZACIONES")
    print("=" * 70)
    print(f"\nEstaciones con datos válidos: {len(df_validos)}")
    print(f"Estación más contaminada: {df_validos.loc[df_validos['pm25'].idxmax(), 'estacion']}")
    print(f"  - PM2.5: {df_validos['pm25'].max():.2f} µg/m³")
    print(f"Estación menos contaminada: {df_validos.loc[df_validos['pm25'].idxmin(), 'estacion']}")
    print(f"  - PM2.5: {df_validos['pm25'].min():.2f} µg/m³")

## 10. Visualización Geográfica con Folium


In [ ]:
# Crear mapa interactivo
def crear_mapa_calidad_aire(df):
    """
    Crea un mapa interactivo con folium mostrando la calidad del aire por estación
    
    Parámetros:
    df (DataFrame): DataFrame con datos de SIATA incluyendo coordenadas e ICA
    
    Retorna:
    folium.Map: Mapa interactivo
    """
    # Filtrar estaciones con coordenadas válidas
    df_mapa = df[(df['latitud'].notna()) & (df['longitud'].notna())]
    
    # Centro del Valle de Aburrá (Medellín)
    centro_mapa = [6.2442, -75.5812]
    
    # Crear mapa
    mapa = folium.Map(location=centro_mapa, zoom_start=11, tiles='OpenStreetMap')
    
    # Crear cluster de marcadores
    marker_cluster = MarkerCluster().add_to(mapa)
    
    # Agregar marcadores para cada estación
    for idx, row in df_mapa.iterrows():
        # Crear popup con información
        popup_html = f"""
        <div style="font-family: Arial; font-size: 12px; min-width: 250px;">
            <h4 style="margin: 0 0 10px 0; color: #333;">{row['estacion']}</h4>
            <table style="width: 100%;">
                <tr>
                    <td><strong>PM2.5:</strong></td>
                    <td>{row['pm25']:.2f} µg/m³</td>
                </tr>
                <tr>
                    <td><strong>ICA:</strong></td>
                    <td>{row['ica'] if row['ica'] else 'N/A'}</td>
                </tr>
                <tr>
                    <td><strong>Categoría:</strong></td>
                    <td style="color: {row['color']}; font-weight: bold;">{row['categoria']}</td>
                </tr>
                <tr>
                    <td><strong>Fecha:</strong></td>
                    <td>{row['fecha_local'].strftime('%Y-%m-%d %H:%M') if pd.notna(row['fecha_local']) else 'N/A'}</td>
                </tr>
                <tr>
                    <td colspan="2" style="padding-top: 10px; border-top: 1px solid #ddd;">
                        <em>{row['recomendacion']}</em>
                    </td>
                </tr>
            </table>
        </div>
        """
        
        # Crear marcador
        folium.Marker(
            location=[row['latitud'], row['longitud']],
            popup=folium.Popup(popup_html, max_width=300),
            icon=folium.Icon(color=row['color'], icon='info-sign'),
            tooltip=row['estacion'][:30] + '...' if len(row['estacion']) > 30 else row['estacion']
        ).add_to(marker_cluster)
    
    # Agregar leyenda
    leyenda_html = """
    <div style="position: fixed; bottom: 50px; left: 50px; width: 200px; 
                background-color: white; padding: 10px; border: 2px solid gray; 
                border-radius: 5px; z-index: 9999; font-size: 12px;">
        <h4 style="margin: 0 0 10px 0;">Leyenda ICA</h4>
        <div style="margin: 5px 0;"><span style="color: green;">■</span> Buena (0-50)</div>
        <div style="margin: 5px 0;"><span style="color: yellow;">■</span> Aceptable (51-100)</div>
        <div style="margin: 5px 0;"><span style="color: orange;">■</span> Moderada (101-150)</div>
        <div style="margin: 5px 0;"><span style="color: red;">■</span> Mala (151-200)</div>
        <div style="margin: 5px 0;"><span style="color: purple;">■</span> Muy Mala (201-300)</div>
        <div style="margin: 5px 0;"><span style="color: maroon;">■</span> Peligrosa (301+)</div>
        <div style="margin: 5px 0;"><span style="color: gray;">■</span> Sin dato</div>
    </div>
    """
    mapa.get_root().html.add_child(folium.Element(leyenda_html))
    
    return mapa

# Crear y mostrar mapa
if not df_siata.empty:
    mapa_siata = crear_mapa_calidad_aire(df_siata)
    display(mapa_siata)
    
    # Guardar mapa como HTML
    mapa_siata.save('mapa_calidad_aire_siata.html')
    print("\n✓ Mapa guardado como 'mapa_calidad_aire_siata.html'")

## 11. Aplicación Interactiva con Filtros


In [ ]:
# Función de consulta interactiva
def consultar_estacion(df, nombre_estacion=None, categoria=None):
    """
    Consulta estaciones con filtros opcionales
    
    Parámetros:
    df (DataFrame): DataFrame con datos
    nombre_estacion (str): Nombre o parte del nombre de la estación
    categoria (str): Categoría de calidad del aire
    
    Retorna:
    DataFrame: Resultados filtrados
    """
    df_filtrado = df.copy()
    
    # Filtrar por nombre de estación
    if nombre_estacion:
        df_filtrado = df_filtrado[df_filtrado['estacion'].str.contains(nombre_estacion, case=False, na=False)]
    
    # Filtrar por categoría
    if categoria:
        df_filtrado = df_filtrado[df_filtrado['categoria'] == categoria]
    
    return df_filtrado

# Demostración de consultas
if not df_siata.empty:
    print("=" * 70)
    print("EJEMPLOS DE CONSULTAS INTERACTIVAS")
    print("=" * 70)
    
    # Ejemplo 1: Estaciones con calidad "Mala"
    print("\n1. Estaciones con calidad del aire 'Mala' o peor:")
    criticas = df_siata[df_siata['categoria'].isin(['Mala', 'Muy Mala', 'Peligrosa'])]
    if len(criticas) > 0:
        display(criticas[['estacion', 'pm25', 'ica', 'categoria', 'recomendacion']])
    else:
        print("   No hay estaciones en categorías críticas")
    
    # Ejemplo 2: Buscar estación por nombre
    print("\n2. Buscar estaciones que contengan 'Centro':")
    centro = consultar_estacion(df_siata, nombre_estacion='Centro')
    if len(centro) > 0:
        display(centro[['estacion', 'pm25', 'ica', 'categoria']])
    else:
        print("   No se encontraron estaciones")
    
    # Ejemplo 3: Top 5 peores estaciones
    print("\n3. Top 5 estaciones con mayor PM2.5:")
    df_validos = df_siata[(df_siata['pm25'] != -9999) & (df_siata['pm25'].notna())]
    top5 = df_validos.nlargest(5, 'pm25')
    display(top5[['estacion', 'pm25', 'ica', 'categoria', 'recomendacion']])

## 12. Sistema de Alertas Tempranas


In [ ]:
# Generar sistema de alertas
def generar_alertas(df):
    """
    Genera alertas tempranas basadas en los niveles de contaminación
    
    Parámetros:
    df (DataFrame): DataFrame con datos de SIATA
    
    Retorna:
    dict: Diccionario con alertas por nivel
    """
    alertas = {
        'roja': [],
        'naranja': [],
        'amarilla': [],
        'verde': []
    }
    
    for idx, row in df.iterrows():
        if row['categoria'] in ['Mala', 'Muy Mala', 'Peligrosa']:
            alertas['roja'].append({
                'estacion': row['estacion'],
                'pm25': row['pm25'],
                'ica': row['ica'],
                'categoria': row['categoria'],
                'recomendacion': row['recomendacion']
            })
        elif row['categoria'] == 'Moderada':
            alertas['naranja'].append({
                'estacion': row['estacion'],
                'pm25': row['pm25'],
                'ica': row['ica']
            })
        elif row['categoria'] == 'Aceptable':
            alertas['amarilla'].append({
                'estacion': row['estacion'],
                'pm25': row['pm25']
            })
        else:
            alertas['verde'].append({
                'estacion': row['estacion'],
                'pm25': row['pm25']
            })
    
    return alertas

# Generar y mostrar alertas
if not df_siata.empty:
    alertas = generar_alertas(df_siata)
    
    print("=" * 70)
    print("SISTEMA DE ALERTAS TEMPRANAS - SIATA")
    print(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 70)
    
    # Alerta Roja
    if alertas['roja']:
        print(f"\n🔴 ALERTA ROJA ({len(alertas['roja'])} estaciones):")
        for alerta in alertas['roja']:
            print(f"  • {alerta['estacion']}")
            print(f"    PM2.5: {alerta['pm25']:.2f} µg/m³ | ICA: {alerta['ica']}")
            print(f"    {alerta['recomendacion']}")
    
    # Alerta Naranja
    if alertas['naranja']:
        print(f"\n🟠 ALERTA NARANJA ({len(alertas['naranja'])} estaciones):")
        for alerta in alertas['naranja'][:5]:  # Mostrar solo las primeras 5
            print(f"  • {alerta['estacion']} - PM2.5: {alerta['pm25']:.2f} µg/m³")
        if len(alertas['naranja']) > 5:
            print(f"  ... y {len(alertas['naranja']) - 5} estaciones más")
    
    # Resumen
    print(f"\n📊 RESUMEN:")
    print(f"  Total estaciones monitoreadas: {len(df_siata)}")
    print(f"  🔴 Alerta Roja: {len(alertas['roja'])}")
    print(f"  🟠 Alerta Naranja: {len(alertas['naranja'])}")
    print(f"  🟡 Alerta Amarilla: {len(alertas['amarilla'])}")
    print(f"  🟢 Verde: {len(alertas['verde'])}")

## 13. Landing Page - Documentación del Proyecto


In [ ]:
# Crear landing page en HTML
def crear_landing_page(df, alertas):
    """
    Genera una landing page HTML con la información del proyecto
    """
    
    html_content = f"""
    <!DOCTYPE html>
    <html lang="es">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>SIATA - Sistema de Monitoreo de Calidad del Aire</title>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 0;
                background-color: #f4f4f4;
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                padding: 20px;
            }}
            header {{
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 40px 0;
                text-align: center;
            }}
            header h1 {{
                margin: 0;
                font-size: 2.5em;
            }}
            header p {{
                font-size: 1.2em;
                margin: 10px 0 0 0;
            }}
            .section {{
                background: white;
                margin: 20px 0;
                padding: 30px;
                border-radius: 8px;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
            }}
            .section h2 {{
                color: #667eea;
                border-bottom: 3px solid #667eea;
                padding-bottom: 10px;
            }}
            .alert-box {{
                padding: 15px;
                margin: 10px 0;
                border-radius: 5px;
                border-left: 5px solid;
            }}
            .alert-roja {{
                background-color: #ffebee;
                border-color: #f44336;
                color: #c62828;
            }}
            .alert-naranja {{
                background-color: #fff3e0;
                border-color: #ff9800;
                color: #e65100;
            }}
            .stats-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 20px;
                margin: 20px 0;
            }}
            .stat-card {{
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 20px;
                border-radius: 8px;
                text-align: center;
            }}
            .stat-card h3 {{
                margin: 0;
                font-size: 2em;
            }}
            .stat-card p {{
                margin: 5px 0 0 0;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
            }}
            th, td {{
                padding: 12px;
                text-align: left;
                border-bottom: 1px solid #ddd;
            }}
            th {{
                background-color: #667eea;
                color: white;
            }}
            .btn {{
                display: inline-block;
                padding: 12px 24px;
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                text-decoration: none;
                border-radius: 5px;
                margin: 10px 5px;
                transition: transform 0.3s;
            }}
            .btn:hover {{
                transform: translateY(-2px);
            }}
            footer {{
                text-align: center;
                padding: 20px;
                background: #333;
                color: white;
                margin-top: 40px;
            }}
        </style>
    </head>
    <body>
        <header>
            <h1>🌍 SIATA - Calidad del Aire</h1>
            <p>Sistema de Monitoreo de Material Particulado PM2.5 en el Valle de Aburrá</p>
        </header>
        
        <div class="container">
            <!-- Problema y Contexto -->
            <div class="section">
                <h2>📋 Problema y Contexto</h2>
                <p><strong>Problema:</strong> La contaminación del aire por material particulado PM2.5 representa un riesgo significativo para la salud pública en el Valle de Aburrá. Las personas vulnerables (niños, adultos mayores y personas con enfermedades respiratorias) necesitan información actualizada para tomar decisiones sobre sus actividades diarias.</p>
                
                <p><strong>Contexto:</strong> El SIATA (Sistema de Alerta Temprana de Medellín y el Valle de Aburrá) opera una de las redes de monitoreo de calidad del aire más sólidas del país, con 43 puntos de monitoreo entre automáticos y manuales, especialmente para material particulado PM2.5 [[22]].</p>
                
                <p><strong>Necesidad:</strong> La Secretaría de Salud Municipal requiere un sistema que permita:</p>
                <ul>
                    <li>Consultar información actualizada en tiempo real</li>
                    <li>Identificar zonas críticas de contaminación</li>
                    <li>Visualizar geográficamente la distribución del contaminante</li>
                    <li>Generar alertas tempranas y recomendaciones</li>
                    <li>Apoyar la toma de decisiones en salud pública</li>
                </ul>
            </div>
            
            <!-- Datos Utilizados -->
            <div class="section">
                <h2>📊 Datos Utilizados</h2>
                <p><strong>Fuente:</strong> SIATA - Sistema de Alerta Temprana de Medellín y el Valle de Aburrá</p>
                <p><strong>API:</strong> https://siata.gov.co/EntregaData1/Datos_SIATA_Aire_AQ_pm25_Last.json</p>
                <p><strong>Variable principal:</strong> Material particulado PM2.5 (µg/m³)</p>
                <p><strong>Cobertura:</strong> {len(df)} estaciones de monitoreo</p>
                <p><strong>Actualización:</strong> Datos horarios en tiempo real</p>
                
                <h3>Variables disponibles:</h3>
                <ul>
                    <li><strong>estacion:</strong> Nombre de la estación de monitoreo</li>
                    <li><strong>pm25:</strong> Concentración de material particulado PM2.5 (µg/m³)</li>
                    <li><strong>ica:</strong> Índice de Calidad del Aire calculado</li>
                    <li><strong>categoria:</strong> Clasificación de la calidad del aire</li>
                    <li><strong>latitud/longitud:</strong> Coordenadas geográficas</li>
                    <li><strong>fecha_local:</strong> Fecha y hora de la medición</li>
                    <li><strong>recomendacion:</strong> Recomendación de salud</li>
                </ul>
            </div>
            
            <!-- Alertas Actuales -->
            <div class="section">
                <h2>🚨 Alertas Actuales</h2>
                <p><strong>Fecha de consulta:</strong> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                
                {f'''<div class="alert-box alert-roja">
                    <h3>🔴 ALERTA ROJA - {len(alertas["roja"])} Estaciones Críticas</h3>
                    <p>Estaciones con calidad del aire Mala, Muy Mala o Peligrosa:</p>
                    <ul>
                        {"".join([f"<li><strong>{a['estacion']}</strong> - PM2.5: {a['pm25']:.2f} µg/m³ (ICA: {a['ica']})<br>{a['recomendacion']}</li>" for a in alertas['roja'][:5]])}
                    </ul>
                </div>''' if alertas['roja'] else '<p>✓ No hay alertas rojas activas</p>'}
                
                {f'''<div class="alert-box alert-naranja">
                    <h3>🟠 ALERTA NARANJA - {len(alertas["naranja"])} Estaciones</h3>
                    <p>Calidad del aire Moderada. Se recomienda precaución.</p>
                </div>''' if alertas['naranja'] else ''}
            </div>
            
            <!-- Estadísticas -->
            <div class="section">
                <h2>📈 Estadísticas Resumen</h2>
                <div class="stats-grid">
                    <div class="stat-card">
                        <h3>{len(df)}</h3>
                        <p>Estaciones Activas</p>
                    </div>
                    <div class="stat-card">
                        <h3>{len(alertas['roja'])}</h3>
                        <p>Alertas Rojas</p>
                    </div>
                    <div class="stat-card">
                        <h3>{len(alertas['naranja'])}</h3>
                        <p>Alertas Naranjas</p>
                    </div>
                    <div class="stat-card">
                        <h3>{df[(df['pm25'] != -9999) & (df['pm25'].notna())]['pm25'].mean():.1f}</h3>
                        <p>Promedio PM2.5 (µg/m³)</p>
                    </div>
                </div>
                
                <h3>Distribución por Categoría:</h3>
                <table>
                    <tr>
                        <th>Categoría</th>
                        <th>Número de Estaciones</th>
                        <th>Porcentaje</th>
                    </tr>
                    {"".join([f"<tr><td>{cat}</td><td>{count}</td><td>{(count/len(df)*100):.1f}%</td></tr>" for cat, count in df['categoria'].value_counts().items()])}
                </table>
            </div>
            
            <!-- Funcionamiento -->
            <div class="section">
                <h2>⚙️ Funcionamiento de la Solución</h2>
                <h3>Flujo de Procesamiento:</h3>
                <ol>
                    <li><strong>Consumo de API:</strong> Conexión HTTP a la API de SIATA para obtener datos en tiempo real</li>
                    <li><strong>Transformación:</strong> Conversión de JSON a DataFrame de pandas</li>
                    <li><strong>Limpieza:</strong> Manejo de valores nulos y datos inválidos (-9999)</li>
                    <li><strong>Cálculo del ICA:</strong> Conversión de µg/m³ a Índice de Calidad del Aire usando estándares EPA/SIATA</li>
                    <li><strong>Clasificación:</strong> Asignación de categorías y colores según rangos ICA</li>
                    <li><strong>Generación de alertas:</strong> Identificación automática de estaciones críticas</li>
                    <li><strong>Visualización:</strong> Mapas interactivos con folium y gráficos estadísticos</li>
                </ol>
                
                <h3>Componentes:</h3>
                <ul>
                    <li><strong>API Consumer:</strong> requests library para consumo de datos</li>
                    <li><strong>Data Processing:</strong> pandas y numpy para transformación</li>
                    <li><strong>Visualización:</strong> matplotlib, seaborn y folium</li>
                    <li><strong>Interactividad:</strong> Widgets de IPython para filtros</li>
                    <li><strong>Mapeo:</strong> Folium con MarkerCluster para visualización geográfica</li>
                </ul>
            </div>
            
            <!-- Utilidad para Toma de Decisiones -->
            <div class="section">
                <h2>💡 Utilidad para la Toma de Decisiones</h2>
                
                <h3>Para la Secretaría de Salud:</h3>
                <ul>
                    <li>Identificación de zonas prioritarias para intervenciones</li>
                    <li>Emisión de alertas sanitarias preventivas</li>
                    <li>Planificación de campañas de salud pública</li>
                    <li>Asignación de recursos a áreas críticas</li>
                </ul>
                
                <h3>Para la Población:</h3>
                <ul>
                    <li>Decidir si realizar actividades al aire libre</li>
                    <li>Proteger a grupos vulnerables (niños, adultos mayores)</li>
                    <li>Planificar rutas de movilidad evitando zonas contaminadas</li>
                    <li>Uso preventivo de mascarillas en zonas críticas</li>
                </ul>
                
                <h3>Para Instituciones Educativas:</h3>
                <ul>
                    <li>Decidir suspensión de actividades deportivas externas</li>
                    <li>Implementar protocolos de protección</li>
                    <li>Informar a padres de familia</li>
                </ul>
            </div>
            
            <!-- Visualizaciones -->
            <div class="section">
                <h2>🗺️ Visualizaciones Disponibles</h2>
                <p>El sistema incluye las siguientes visualizaciones:</p>
                <ul>
                    <li><strong>Mapa Interactivo:</strong> Visualización geográfica de estaciones con códigos de color según ICA (archivo: mapa_calidad_aire_siata.html)</li>
                    <li><strong>Histogramas:</strong> Distribución de concentraciones PM2.5</li>
                    <li><strong>Gráficos de barras:</strong> Comparación entre estaciones</li>
                    <li><strong>Boxplots:</strong> Análisis estadístico de distribución</li>
                    <li><strong>Tablas dinámicas:</strong> Consultas filtradas por categoría o ubicación</li>
                </ul>
                
                <a href="mapa_calidad_aire_siata.html" class="btn" target="_blank">Ver Mapa Interactivo</a>
            </div>
            
            <!-- Conclusiones -->
            <div class="section">
                <h2>✅ Conclusiones</h2>
                <ul>
                    <li>El sistema permite monitoreo en tiempo real de la calidad del aire en el Valle de Aburrá</li>
                    <li>La visualización geográfica facilita la identificación de patrones espaciales de contaminación</li>
                    <li>Las alertas automáticas permiten respuesta rápida ante episodios críticos</li>
                    <li>La solución es escalable y puede integrarse con otros sistemas de la Secretaría de Salud</li>
                    <li>Los datos históricos pueden usarse para análisis predictivos y modelamiento</li>
                </ul>
            </div>
        </div>
        
        <footer>
            <p>SIATA - Sistema de Alerta Temprana de Medellín y el Valle de Aburrá</p>
            <p>Proyecto de Análisis de Datos - Calidad del Aire PM2.5</p>
            <p>© {datetime.now().year}</p>
        </footer>
    </body>
    </html>
    """
    
    # Guardar archivo HTML
    with open('landing_page_siata.html', 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print("✓ Landing page creada: landing_page_siata.html")
    print("  Abra el archivo en su navegador para visualizar")

# Crear landing page
if not df_siata.empty:
    crear_landing_page(df_siata, alertas)

## 14. Resumen de Evidencias Entregadas


In [ ]:
print("=" * 70)
print("RESUMEN DE EVIDENCIAS DEL PROYECTO")
print("=" * 70)

evidencias = [
    ("✓", "Consumo de API SIATA", "Datos PM2.5 en tiempo real"),
    ("✓", "Exploración de datos", f"{len(df_siata)} estaciones analizadas"),
    ("✓", "Cálculo del ICA", "Conversión PM2.5 → ICA con categorías"),
    ("✓", "Visualizaciones estadísticas", "Histogramas, boxplots, barras"),
    ("✓", "Mapa interactivo", "mapa_calidad_aire_siata.html"),
    ("✓", "Sistema de alertas", f"{len(alertas['roja'])} alertas rojas, {len(alertas['naranja'])} naranjas"),
    ("✓", "Consultas interactivas", "Filtros por estación y categoría"),
    ("✓", "Landing page", "landing_page_siata.html")
]

for icon, item, detalle in evidencias:
    print(f"\n{icon} {item}")
    print(f"   → {detalle}")

print("\n" + "=" * 70)
print("ARCHIVOS GENERADOS")
print("=" * 70)
print("1. notebook_siata_calidad_aire.ipynb - Cuaderno completo")
print("2. mapa_calidad_aire_siata.html - Mapa interactivo")
print("3. landing_page_siata.html - Documentación del proyecto")
print("\n" + "=" * 70)
print("PROYECTO COMPLETADO EXITOSAMENTE")
print("=" * 70)